# 4 — Run Full Pipeline

Orchestrates all 3 refresh steps in sequence:
1. Fetch transactions from SBIS API → `transactions` Delta table
2. Build daily sales from transactions → `daily_sales` Delta table
3. Build product sales from transactions → `product_sales` Delta table

**Usage:** Set the date range below, then Run All.

In [ ]:
# ── Date Range Config ──────────────────────────────────────────────

from datetime import datetime

today = datetime.now()
START_DATE = datetime(today.year, today.month, today.day, 0, 0, 0)
END_DATE   = datetime(today.year, today.month, today.day, 23, 59, 59)

DATE_FROM = START_DATE.strftime("%Y-%m-%d")
DATE_TO   = END_DATE.strftime("%Y-%m-%d")

print(f"Pipeline date range: {DATE_FROM} → {DATE_TO}")

In [ ]:
# ── Step 1: Refresh Transactions ──────────────────────────────────

from pipeline.sbis_api import authenticate, get_sales_points, fetch_orders
from pipeline.transforms import build_transactions, save_delta, TRANSACTIONS_SCHEMA
from pipeline.config import TRANSACTIONS_TABLE

print("=" * 60)
print("STEP 1: Fetching transactions from SBIS API")
print("=" * 60)

sid = authenticate()
points = get_sales_points(sid)
raw = fetch_orders(sid, points, START_DATE, END_DATE)
transactions_df = build_transactions(raw)

save_delta(transactions_df, TRANSACTIONS_TABLE, TRANSACTIONS_SCHEMA, DATE_FROM, DATE_TO)
print(f"Transactions: {len(transactions_df):,} rows\n")

In [ ]:
# ── Step 2: Refresh Daily Sales ───────────────────────────────────

from pipeline.transforms import load_transactions, build_daily_sales, DAILY_SALES_SCHEMA
from pipeline.config import DAILY_SALES_TABLE

print("=" * 60)
print("STEP 2: Building daily sales")
print("=" * 60)

txn = load_transactions(date_from=DATE_FROM, date_to=DATE_TO)
daily_df = build_daily_sales(txn)

save_delta(daily_df, DAILY_SALES_TABLE, DAILY_SALES_SCHEMA, DATE_FROM, DATE_TO)
print(f"Daily sales: {len(daily_df):,} rows\n")

In [ ]:
# ── Step 3: Refresh Product Sales ─────────────────────────────────

from pipeline.transforms import build_product_sales, PRODUCT_SALES_SCHEMA
from pipeline.config import PRODUCT_SALES_TABLE

print("=" * 60)
print("STEP 3: Building product sales")
print("=" * 60)

product_df = build_product_sales(txn)

save_delta(product_df, PRODUCT_SALES_TABLE, PRODUCT_SALES_SCHEMA, DATE_FROM, DATE_TO)
print(f"Product sales: {len(product_df):,} rows\n")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────

print("Pipeline complete!")
print(f"  Date range:    {DATE_FROM} → {DATE_TO}")
print(f"  Transactions:  {len(transactions_df):,} rows")
print(f"  Daily sales:   {len(daily_df):,} rows")
print(f"  Product sales: {len(product_df):,} rows")